In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
# Connect to Google Drive
drive.mount('/content/drive')

In [ ]:
# Read the D3.csv
file_path = '/content/drive/My Drive/Datasets/D3.csv'
df = pd.DataFrame(pd.read_csv(file_path))

In [ ]:
# Extract the values
m = len(df)  # number of training examples
y = df.values[:,3]
print('Number of training examples m =', m)
df.head()

In [ ]:
# Cost function (https://github.com/HamedTabkhi/Intro-to-ML/blob/main/IntroCodes/LinearRegression.ipynb)
def compute_cost(X, y, theta):
    """
    Compute cost for linear regression.

    Parameters:
    X : 2D array where each row represents the training example and each column represent the feature
        m = number of training examples
        n = number of features (including X_0 column of ones)
    y : 1D array of labels/target values for each training example. dimension(m)
    theta : 1D array of fitting parameters or weights. Dimension (n)

    Returns:
    J : Scalar value, the cost
    """
    predictions = X.dot(theta)
    errors = np.subtract(predictions, y)
    sqrErrors = np.square(errors)
    J = 1 / (2 * m) * np.sum(sqrErrors)
    return J

In [ ]:
# Gradient descent function (https://github.com/HamedTabkhi/Intro-to-ML/blob/main/IntroCodes/LinearRegression.ipynb)
def gradient_descent(X, y, theta, alpha, iterations):
    """
    Compute the optimal parameters using gradient descent for linear regression.

    Parameters:
    X : 2D array where each row represents the training example and each column represents the feature
        m = number of training examples
        n = number of features (including X_0 column of ones)
    y : 1D array of labels/target values for each training example. dimension(m)
    theta : 1D array of fitting parameters or weights. Dimension (n)
    alpha : Learning rate (scalar)
    iterations : Number of iterations (scalar)

    Returns:
    theta : Updated values of fitting parameters or weights after 'iterations' iterations. Dimension (n)
    cost_history : Array containing the cost for each iteration. Dimension (iterations)
    """

    m = len(y)  # Number of training examples
    cost_history = np.zeros(iterations)

    for i in range(iterations):
        predictions = X.dot(theta)
        errors = np.subtract(predictions, y)
        sum_delta = (alpha / m) * X.transpose().dot(errors)
        theta -= sum_delta
        cost_history[i] = compute_cost(X, y, theta)

    return theta, cost_history

In [ ]:
# Training
iterations = 1000
alphas = [0.1, 0.05, 0.01]
variables = ['X1', 'X2', 'X3']

results = {}

for var in variables:
    x = df[var].values
    X_0 = np.ones((m, 1))
    X_1 = x.reshape(m, 1)
    X = np.hstack((X_0, X_1))
    results[var] = {}
    for alpha in alphas:
        theta0 = np.zeros(2)
        theta, cost_history = gradient_descent(X, y, theta0, alpha, iterations)
        results[var][alpha] = {
            'theta': theta,
            'cost_history': cost_history
        }
        print(f"{var:>3s} | alpha={alpha:<4} | theta0={theta[0]: .4f}  theta1={theta[1]: .4f} "
              f"| final cost={cost_history[-1]:.4f} ")
    print()

In [ ]:
#Plotting
fig, axes = plt.subplots(3, 2, figsize=(13, 15))

for i, var in enumerate(variables):
    x = df[var].values
    best_alpha = 0.1
    theta = results[var][best_alpha]['theta']

    # Left side - Linear Regression
    ax = axes[i, 0]
    ax.scatter(x, y, color='red', marker='+', label='Training data')
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = theta[0] + theta[1] * x_line
    ax.plot(x_line, y_line, color='green', linewidth=2, label=f'Linear Regression (alpha={best_alpha})')
    ax.set_xlabel(var)
    ax.set_ylabel('Y')
    ax.set_title(f'Linear Regression Fit for {var}')
    ax.grid(True)
    ax.legend()

    # Right side - Cost function
    ax = axes[i, 1]
    for alpha in alphas:
        ch = results[var][alpha]['cost_history']
        ax.plot(range(1, iterations + 1), ch, label=f'alpha={alpha}')
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Cost J(theta)')
    ax.set_title(f'{var}: Cost vs. Iteration for different learning rates')
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()